In [ ]:
import pandas as pd
import requests
import time
import json
import os

# deduplicate the books dataset first
INPUT_FILE = r"C:\Users\LOQ\project_books\.venv\datas\processed_books_dataset.csv"
OUTPUT_FILE = "openlibrary_metadata.csv"

MAX_RETRIES = 3
REQUEST_DELAY = 1

if os.path.exists(OUTPUT_FILE):
    results_df = pd.read_csv(
        OUTPUT_FILE,
        dtype={
            "google_id": "string"
        }
    )

    processed_ids = set(
        results_df["google_id"]
        .dropna()
        .astype(str)
    )
    print(
        f"Loaded {len(results_df)} previously processed books."
    )

else:
    results_df = pd.DataFrame()
    processed_ids = set()
    print("No previous progress found. Starting from the beginning.")

No previous progress found. Starting from the beginning.


In [35]:
books_df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "google_id": "string",
        "isbn_10": "string",
        "isbn_13": "string"
    }
)

print(books_df.shape)

(7191, 16)


In [ ]:
def clean_isbn(isbn):
    if pd.isna(isbn):
        return None
    
    isbn = str(isbn).strip()
    if isbn.endswith(".0"):
        isbn = isbn[:-2]

    isbn = isbn.replace("-", "")
    isbn = isbn.replace(" ", "")

    if isbn == "":
        return None

    return isbn

In [ ]:
def make_request(url):

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(
                url,
                timeout=20,
                headers={
                    "User-Agent": "project_books/1.0"
                }
            )

            if response.status_code == 200:
                return response
            if response.status_code == 404:
                return None
            if response.status_code in [429, 500, 502, 503, 504]:
                wait = 2 ** attempt

                print(
                    f"Temporary error "
                    f"{response.status_code}. "
                    f"Waiting {wait} seconds..."
                )
                time.sleep(wait)
                continue

            print(
                f"HTTP error: {response.status_code}"
            )
            return None

        except requests.exceptions.Timeout:
            print(
                f"Timeout. "
                f"Attempt {attempt + 1}/{MAX_RETRIES}"
            )

            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:

            print("Request failed:", e)

            time.sleep(2 ** attempt)

    return None

In [ ]:
def get_work_from_isbn(isbn):
    isbn = clean_isbn(isbn)

    if not isbn:
        return None

    url = (
        f"https://openlibrary.org/isbn/"
        f"{isbn}.json"
    )

    response = make_request(url)
    if response is None:
        return None

    data = response.json()
    works = data.get("works")

    if not works:
        return None

    return works[0].get("key")

In [ ]:
def search_openlibrary(title, author=None):

    if pd.isna(title):
        return None

    title = str(title).strip()

    if not title:
        return None

    url = "https://openlibrary.org/search.json"

    params = {
        "title": title,
        "limit": 10
    }

    if author and not pd.isna(author):
        params["author"] = str(author)

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=20,
                headers={
                    "User-Agent": "project_books/1.0"
                }
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in [429, 500, 502, 503, 504]:

                wait = 2 ** attempt

                print(
                    f"Temporary error "
                    f"{response.status_code}. "
                    f"Waiting {wait} seconds..."
                )

                time.sleep(wait)
                continue

            return None

        except requests.exceptions.Timeout:

            print(
                f"Search timeout. "
                f"Attempt {attempt + 1}/{MAX_RETRIES}"
            )

            time.sleep(2 ** attempt)

        except requests.exceptions.RequestException as e:

            print("Search failed:", e)

            time.sleep(2 ** attempt)

    return None

In [41]:
def normalize_text(text):

    if pd.isna(text):
        return ""

    return " ".join(
        str(text).lower().strip().split()
    )

In [ ]:
def title_matches(
    google_title,
    openlibrary_title
):
    google_title = normalize_text(
        google_title
    )
    openlibrary_title = normalize_text(
        openlibrary_title
    )

    if not google_title or not openlibrary_title:
        return False

    return (
        google_title == openlibrary_title
        or google_title in openlibrary_title
        or openlibrary_title in google_title
    )

In [43]:
def author_matches(
    google_author,
    openlibrary_authors
):

    if not google_author:
        return False

    if not openlibrary_authors:
        return False

    google_author = normalize_text(
        google_author
    )

    for author in openlibrary_authors:

        author = normalize_text(author)

        if (
            author in google_author
            or google_author in author
        ):
            return True

    return False

In [ ]:
def choose_search_result(
    google_title,
    google_author,
    search_data
):

    if not search_data:
        return None

    docs = search_data.get(
        "docs",
        []
    )

    # First priority:
    # title + author match
    for doc in docs:
        title_ok = title_matches(
            google_title,
            doc.get("title")
        )
        author_ok = author_matches(
            google_author,
            doc.get("author_name", [])
        )
        if title_ok and author_ok:
            return doc

    # Second priority:
    # exact/similar title match

    for doc in docs:
        if title_matches(
            google_title,
            doc.get("title")
        ):
            return doc
    return None

In [ ]:
def get_work(work_key):
    if not work_key:
        return None

    url = (
        f"https://openlibrary.org"
        f"{work_key}.json"
    )
    response = make_request(url)

    if response is None:
        return None

    return response.json()

In [ ]:
def extract_description(work_data):

    if not work_data:
        return None

    description = work_data.get(
        "description"
    )
    if isinstance(description, dict):

        return description.get(
            "value"
        )
    if isinstance(description, str):
        return description

    return None

In [47]:
def extract_subjects(work_data):

    if not work_data:
        return []

    subjects = work_data.get(
        "subjects",
        []
    )

    if not isinstance(subjects, list):
        return []

    return subjects

In [ ]:
def get_openlibrary_data(row):
    title = row["title"]
    authors = row["authors"]

    isbn_13 = clean_isbn(
        row["isbn_13"]
    )
    isbn_10 = clean_isbn(
        row["isbn_10"]
    )

    work_key = None
    match_method = None

    if isbn_13:
        print(
            f"{title} → trying ISBN-13 "
            f"{isbn_13}")
        work_key = get_work_from_isbn(
            isbn_13)

        if work_key:
            match_method = "isbn_13"

    if work_key is None and isbn_10:
        print(
            f"{title} → trying ISBN-10 "
            f"{isbn_10}")

        work_key = get_work_from_isbn(
            isbn_10)

        if work_key:
            match_method = "isbn_10"

    if work_key is None:
        print(
            f"{title} → trying "
            f"title + author"
        )
        search_data = search_openlibrary(
            title,
            authors
        )
        selected = choose_search_result(
            title,
            authors,
            search_data
        )

        if selected:
            work_key = selected.get(
                "key"
            )
            if work_key:
                match_method = "title_author"


    if work_key is None:
        print(
            f"{title} → NOT FOUND"
        )

        return {
            "google_id": row["google_id"],
            "title": title,
            "openlibrary_work_key": None,
            "openlibrary_description": None,
            "openlibrary_subjects": None,
            "match_method": "not_found"
        }

    print(
        f"{title} → {work_key}"
    )
    work_data = get_work(
        work_key
    )

    if work_data is None:
        return {
            "google_id": row["google_id"],
            "title": title,
            "openlibrary_work_key": work_key,
            "openlibrary_description": None,
            "openlibrary_subjects": None,
            "match_method": "work_fetch_failed"
        }

    description = extract_description(
        work_data
    )
    subjects = extract_subjects(
        work_data
    )
    return {
        "google_id": row["google_id"],
        "title": title,
        "openlibrary_work_key": work_key,
        "openlibrary_description": description,
        "openlibrary_subjects": json.dumps(
            subjects,
            ensure_ascii=False
        ),
        "match_method": match_method
    }

In [ ]:
for i, (_, row) in enumerate(books_df.iterrows()):
    google_id = str(row["google_id"])
    if google_id in processed_ids:

        print(
            f"Skipping {row['title']} "
            f"(already processed)"
        )
        continue

    try:
        result = get_openlibrary_data(row)

        # Add result to dataframe
        result_df = pd.DataFrame([result])

        results_df = pd.concat(
            [results_df, result_df],
            ignore_index=True
        )

        processed_ids.add(google_id)

        results_df.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"Saved progress: "
            f"{len(results_df)} books"
        )

    except Exception as e:
        print(
            f"ERROR processing "
            f"{row['title']}: {e}"
        )

    time.sleep(1)

The Face of Another → trying title + author
The Face of Another → /works/OL8037368W
Saved progress: 1 books
Generations of Winter → trying title + author
Generations of Winter → NOT FOUND
Saved progress: 2 books
The Street of Crocodiles → trying ISBN-13 9788087830277
The Street of Crocodiles → trying ISBN-10 808783027X
The Street of Crocodiles → trying title + author
Search failed: HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?title=The+Street+of+Crocodiles&limit=10&author=Bruno+Schulz (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))
The Street of Crocodiles → /works/OL2510290W
Saved progress: 3 books
Kokoro → trying ISBN-13 9781462900251
Kokoro → /works/OL25366592W
Saved progress: 4 books
Sense and Sensibility - Illustrated Edition → trying ISBN-13 9788826096476
Sense and Sensibility - Illustrated Edition → trying ISBN-10 8826096473
Request failed: HTTPSC

KeyboardInterrupt: 